<a href="https://colab.research.google.com/github/YaS16s/Deep-learning/blob/main/divingyolo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install the ultralytics package if it's not already installed
!pip install ultralytics

from ultralytics import YOLO
import cv2

In [ ]:
from google.colab.patches import cv2_imshow

model = YOLO("yolov8n.pt")
img=cv2.imread("/content/census-information-city-composition_23-2148993128.avif")
results= model(img)
for r in results:
    annotated = r.plot()

cv2_imshow(annotated)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
from google.colab.patches import cv2_imshow

model = YOLO("yolov8n.pt")
img=cv2.imread("/content/census-information-city-composition_23-2148993128.avif")
results= model(img)
for r in results:
    annotated = r.plot()

cv2_imshow(annotated)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
from google.colab.patches import cv2_imshow

model = YOLO("yolov8n.pt")
img=cv2.imread("/content/census-information-city-composition_23-2148993128.avif")
results= model(img)
for r in results:
    annotated = r.plot()

cv2_imshow(annotated)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
model= YOLO("yolov9c.pt")
# Adjust confidence (conf) and Intersection over Union (iou) thresholds
# Lower 'conf' will show detections the model is less certain about.
# Lower 'iou' will allow more overlapping bounding boxes.
results= model(img, conf = 0.10 , iou = 0.10)
for r in results:
  annotated= r.plot()

cv2_imshow(annotated)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
# Download YOLOv9c model weights
!wget -q https://github.com/ultralytics/assets/releases/download/v8.1.0/yolov9c.pt
!ls -lh /content/

### Confidence (conf)

In object detection, **confidence** (often denoted as `conf` or `objectness score`) is a probability score, typically ranging from 0 to 1, that a model assigns to a predicted bounding box.

It represents two things:
1.  **Probability of an object existing:** How likely it is that *any* object is present within the predicted bounding box.
2.  **Probability of a specific class:** How likely it is that the detected object belongs to a particular class (e.g., 'bird', 'person').

Mathematically, for each predicted bounding box, the model outputs a confidence score for each possible class. If the confidence score for a certain class is below a set `conf` threshold, that detection is typically discarded.

**Example:** If a model predicts a bounding box with a confidence score of 0.85 for the class 'bird', it means the model is 85% confident that there's a bird in that box. If our `conf` threshold is 0.7, this detection would be kept. If it was 0.6, it would also be kept. But if our `conf` threshold was 0.9, this detection would be discarded.

### Intersection over Union (IoU)

**Intersection over Union (IoU)** is a metric used to quantify the overlap between two bounding boxes: a **predicted bounding box** and a **ground truth bounding box** (the true location of the object).

It is defined by the formula:

$$\text{IoU} = \frac{\text{Area of Intersection}}{\text{Area of Union}}$$


Where:
*   **Area of Intersection:** The area where the predicted box and the ground truth box overlap.
*   **Area of Union:** The total area covered by both bounding boxes combined (Intersection + non-overlapping areas of both boxes).


**How IoU is used:**

1.  **Evaluation Metric:** During model training and evaluation, IoU is used to determine if a detection is considered a True Positive. If the IoU between a predicted box and a ground truth box exceeds a certain threshold (e.g., 0.5), the prediction is considered correct.

2.  **Non-Maximum Suppression (NMS):** In post-processing, after a model generates many bounding boxes for an image, NMS uses IoU to eliminate redundant or overlapping detections for the *same object*. The process generally works as follows:
    *   Select the bounding box with the highest confidence score.
    *   Compare this box with all other bounding boxes using IoU.
    *   Any other bounding box that has an IoU with the highest confidence box above a certain `iou` threshold (e.g., 0.45) is suppressed (removed), as it's likely detecting the same object.
    *   Repeat until all boxes have been processed.

**Example:** If the model predicts two bounding boxes for the same bird, one with a confidence of 0.9 and another with 0.8. If the IoU between these two boxes is 0.7 (and our `iou` threshold for NMS is 0.5), the box with 0.8 confidence will be suppressed, keeping only the one with 0.9 confidence.

In [ ]:
!pip install ultralytics
import math
import cv2
from ultralytics import YOLO
from google.colab.patches import cv2_imshow
from deep_sort_realtime.deepsort_tracker import DeepSort

model = YOLO("yolov8n.pt")
tracker = Deepsort(max_age=30)
cap = cv2.VideoCapture("/content/From Main Klickpin CF- pakistan - 1mhjmlvQJ.mp4")

# Initialize object tracking variables outside the loop
objects = {}
objects_id = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)

    nobjects = {}
    for r in results:
        boxes = r.boxes
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2

            matched = False

            for obj_id, (px, py) in objects.items():
                distance = math.hypot(cx - px, cy - py)

                if distance < 50:
                    nobjects[obj_id] = (cx, cy)
                    matched = True
                    # Draw text and rectangle for matched object
                    cv2.putText(frame, str(obj_id), (x1, y1 - 10), cv2.FONT_HERSHEY_COMPLEX, 0.5, (255, 0, 0), 2)
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
                    break

            if not matched:
                objects_id += 1
                nobjects[objects_id] = (cx, cy)
                # Draw text and rectangle for new object
                cv2.putText(frame, str(objects_id), (x1, y1 - 10), cv2.FONT_HERSHEY_COMPLEX, 0.5, (255, 0, 0), 2)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)

    objects = nobjects # Update the objects for the next frame

    cv2_imshow(frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

Deepsort

In [ ]:
!pip install ultralytics # Install ultralytics
!pip install deep_sort_realtime # Install deep_sort_realtime

import cv2
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
from google.colab.patches import cv2_imshow # Import cv2_imshow for Colab

model = YOLO("yolov8n.pt")
tracker = DeepSort(max_age=30)

cap = cv2.VideoCapture("/content/From Main Klickpin CF- pakistan - 1mhjmlvQJ.mp4")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)

    detections = []

    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            cls = int(box.cls[0])

            detections.append(([x1, y1, x2-x1, y2-y1], conf, cls))

    tracks = tracker.update_tracks(detections, frame=frame)

    for track in tracks:
        if not track.is_confirmed():
            continue

        track_id = track.track_id
        l, t, w, h = map(int, track.to_ltrb())

        cv2.rectangle(frame, (l,t), (l+w, t+h), (0,255,0), 2)
        cv2.putText(frame, f"ID {track_id}", (l,t-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    cv2_imshow(frame) # Use cv2_imshow instead of cv2.imshow

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()